# Descriptor-level distribution results as a LaTeX table

Scegliere **DATASET, MODEL e SPLIT** nella prima cella di codice ed eseguire tutte
le celle. Il notebook restituisce la tabella per descrittore del report scelto,
leggendo direttamente il relativo `details/scores/metrics.json`.

La configurazione iniziale è **QM9 / MiDi / train**, come nel report indicato.
`METRIC = "wasserstein"` riproduce la tabella **Wasserstein-1 per descrittore**:
descrittore, unità, Base, MMFF, Atom details e minimo medio. Ogni cella riporta
**media ± SD campionaria (n−1) e numero di seed**. `count` corrisponde a
`conteggio` nel report Markdown; `e` indica la carica elementare.

Si possono selezionare anche `METRIC = "tvd"` oppure `"js_distance"` per ottenere
una tabella con la stessa struttura e i relativi descrittori disponibili.
Queste due distanze sono adimensionali. Jensen–Shannon indica la **distanza**
(radice della divergenza, logaritmi in base 2). Nessuna metrica viene convertita
in percentuale o ricalcolata; si usano le statistiche salvate dall'analisi.

Il minimo è determinato prima dell'arrotondamento, con la stessa tolleranza del
report; si riportano anche eventuali parità. È un confronto descrittivo, senza
implicazioni sulla significatività statistica.

Resta attivo il requisito di **tre seed** (`REQUIRED_SEEDS = 3`). Metriche con
un altro conteggio sono mostrate come `/`, insieme al conteggio effettivo, e
non partecipano al minimo. Impostare `REQUIRED_SEEDS = None` per mostrare tutti
i conteggi. Un solo seed ha SD non disponibile, mai sostituita con zero.

Gli split `val` e `valid` sono alias di `validation`. Se la combinazione non
esiste, il messaggio di errore elenca quelle disponibili. Le analisi marcate
come incomplete vengono rifiutate. `RESULTS_ROOT` può essere un percorso
assoluto se il notebook è eseguito da un'altra cartella.

L'output visibile è stato generato dai JSON locali. `OUTPUT_PATH` permette di
salvare anche un `.tex`; per impostazione predefinita si stampa solo il LaTeX.
Il documento LaTeX richiede:

```latex
\usepackage{booktabs,graphicx}
\providecommand{\nlvalpm}[2]{\ensuremath{#1\,{\scriptstyle\pm}\,#2}}
```


In [ ]:
from pathlib import Path

RESULTS_ROOT = Path("outputs/distribution_comparison")

# Select one result folder. Accepted split aliases: val/valid -> validation.
DATASET = "qm9"
MODEL = "midi"
SPLIT = "train"

# The Markdown report's per-descriptor table uses Wasserstein-1.
# Other choices: "tvd", "js_distance" (dimensionless distances).
METRIC = "wasserstein"
DECIMALS = 6
REPRESENTATIONS = (("base", "Base"), ("mmff", "MMFF"), ("atom_details", "Atom details"))

# Keep the three-seed rule. Set None to show any available seed count.
REQUIRED_SEEDS = 3

# Optional .tex export; None prints the table without creating extra files.
OUTPUT_PATH = None  # For example: Path("descriptor_table.tex").


In [ ]:
import json
import math


def normalize_split(value):
    if value is None:
        return "full"
    return {"valid": "validation", "val": "validation", "training": "train"}.get(value, value)


def normalize_dataset(value):
    return {"zinc": "zinc250k", "zinc_mmff": "zinc250k_mmff",
            "zinc_atom_details": "zinc250k_atom_details"}.get(value, value)


def load_selected_result(root, dataset, model, split):
    root = Path(root).expanduser().resolve()
    split = normalize_split(split)
    folders = ("validation", "valid", "val") if split == "validation" else (split,)
    candidates = [root / dataset / model / folder / "details/scores/metrics.json" for folder in folders]
    available = [path for path in candidates if path.is_file()]
    if not available:
        choices = sorted("/".join(path.relative_to(root).parts[:3])
                         for path in root.glob("*/*/*/details/scores/metrics.json"))
        raise FileNotFoundError(f"No score file for {dataset}/{model}/{split} under {root}. "
                                f"Available combinations: {', '.join(choices) or 'none'}")
    if len(available) > 1:
        raise ValueError(f"Multiple score files for the selected split: {available}")
    path = available[0]
    report = path.parents[2] / "REPORT.md"
    if report.is_file() and report.read_text(encoding="utf-8").startswith(
            ("# Analisi non completata", "# Analisi in corso")):
        raise ValueError(f"Selected analysis is incomplete: {report}")
    document = json.loads(path.read_text(encoding="utf-8"))
    metadata = document["metadata"]
    identity = (normalize_dataset(metadata["reference_dataset"]), metadata["model"],
                normalize_split(metadata["reference_split"]))
    if identity != (normalize_dataset(dataset), model, split):
        raise ValueError(f"Selected dataset/model/split does not match metadata in {path}: {identity}")
    if not isinstance(document.get("representations"), dict):
        raise ValueError(f"Missing representation scores in {path}")
    return {"path": path, "scope": (dataset, model, split), "document": document}


def latex_escape(text):
    replacements = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$",
                    "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}",
                    "~": r"\textasciitilde{}", "^": r"\textasciicircum{}"}
    return "".join(replacements.get(char, char) for char in str(text))


def descriptor_stat(document, representation, descriptor, metric):
    return (document["representations"].get(representation, {}).get("summary", {})
            .get("descriptors", {}).get(descriptor, {}).get(metric, {}))


def eligible_mean(stat):
    count = stat.get("n", 0)
    if not isinstance(count, int) or isinstance(count, bool) or count < 0:
        raise ValueError(f"Invalid seed count: {count}")
    mean = stat.get("mean")
    if (count == 0 or mean is None or not math.isfinite(mean)
            or (REQUIRED_SEEDS is not None and count != REQUIRED_SEEDS)):
        return None
    return mean


def format_descriptor_stat(stat):
    count = stat.get("n", 0)
    mean = eligible_mean(stat)
    suffix = rf" {{\scriptsize ($n={count}$)}}"
    if mean is None:
        return "/" + suffix
    std = stat.get("std")
    if std is None or count < 2 or not math.isfinite(std):
        return rf"\ensuremath{{{mean:.{DECIMALS}f}\;(\mathrm{{SD\ n/a}})}}" + suffix
    if std < 0:
        raise ValueError(f"Negative SD: {std}")
    return rf"\nlvalpm{{{mean:.{DECIMALS}f}}}{{{std:.{DECIMALS}f}}}" + suffix


def make_descriptor_table(result, metric="wasserstein"):
    metric_titles = {"wasserstein": "Wasserstein-1", "tvd": "TVD",
                     "js_distance": "Jensen--Shannon distance"}
    if metric not in metric_titles:
        raise ValueError(f"Unknown metric {metric!r}; choose {tuple(metric_titles)}")
    document = result["document"]
    fields = sorted({field
                     for name, _ in REPRESENTATIONS
                     for field, metrics in document["representations"].get(name, {}).get("summary", {}).get("descriptors", {}).items()
                     if metrics.get(metric, {}).get("n", 0) > 0})
    if not fields:
        raise ValueError(f"No descriptor-level {metric} results available in {result['path']}")
    dataset, model, split = result["scope"]
    dataset_label = {"qm9": "QM9", "gdb13": "GDB13", "zinc": "ZINC", "zinc250k": "ZINC"}.get(dataset, dataset)
    model_label = {"midi": "MiDi", "d4": "D4"}.get(model, model)
    lines = [
        "% Source: " + result["path"].as_posix(),
        r"\begin{table*}[t]",
        r"\centering",
        r"\small",
        r"\setlength{\tabcolsep}{4pt}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{@{}ll" + "c" * len(REPRESENTATIONS) + r"l@{}}",
        r"\toprule",
        "Descriptor & Unit & " + " & ".join(latex_escape(label) for _, label in REPRESENTATIONS)
        + r" & Lowest mean \\",
        r"\midrule",
    ]
    for descriptor in fields:
        stats = {name: descriptor_stat(document, name, descriptor, metric) for name, _ in REPRESENTATIONS}
        means = {name: eligible_mean(stat) for name, stat in stats.items()}
        means = {name: mean for name, mean in means.items() if mean is not None}
        minimum = min(means.values()) if means else None
        best = [label for name, label in REPRESENTATIONS
                if name in means and math.isclose(means[name], minimum, abs_tol=1e-12, rel_tol=0)]
        unit = ("e" if descriptor in {"formal_charges", "molecular_charge"} else "count") if metric == "wasserstein" else "dimensionless"
        cells = [latex_escape(descriptor), unit]
        cells.extend(format_descriptor_stat(stats[name]) for name, _ in REPRESENTATIONS)
        cells.append(latex_escape(", ".join(best)) if best else "/")
        lines.append(" & ".join(cells) + r" \\")
    title = f"{metric_titles[metric]} per descriptor: {model_label}, {dataset_label}, reference split {split}."
    lines.extend([
        r"\bottomrule",
        r"\end{tabular}%",
        r"}",
        r"\caption{" + latex_escape(title) + r" Entries are mean $\pm$ sample SD across training seeds; "
        r"$n$ is the number of seeds with a defined distance. Lower is better. "
        "The lowest mean is descriptive and does not imply statistical significance. "
        + (f"Metrics with a seed count other than {REQUIRED_SEEDS} are shown as /. " if REQUIRED_SEEDS is not None else "")
        + ("Wasserstein distances retain descriptor units; e denotes elementary charge. " if metric == "wasserstein" else "Distances are on their original scale, without percentage conversion. ")
        + ("Jensen--Shannon is the square root of the divergence with base-2 logarithms. " if metric == "js_distance" else "")
        + "SD is undefined for a single seed; missing values are /.}",
        r"\label{tab:descriptors:" + ":".join((dataset, model, split, metric)) + "}",
        r"\end{table*}",
    ])
    return "\n".join(lines)


In [ ]:
selected_result = load_selected_result(RESULTS_ROOT, DATASET, MODEL, SPLIT)
print("Selected:", " / ".join(selected_result["scope"]))
print("Metric:", METRIC)
print("Source:", selected_result["path"])


Selected: qm9 / midi / train
Metric: wasserstein
Source: /mnt/projects/dpg/hit/u0074/repository/D4-distance-diffusion/outputs/distribution_comparison/qm9/midi/train/details/scores/metrics.json


In [ ]:
latex_output = make_descriptor_table(selected_result, metric=METRIC)
print(latex_output)

if OUTPUT_PATH is not None:
    output = Path(OUTPUT_PATH)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(latex_output + "\n", encoding="utf-8")


% Source: /mnt/projects/dpg/hit/u0074/repository/D4-distance-diffusion/outputs/distribution_comparison/qm9/midi/train/details/scores/metrics.json
\begin{table*}[t]
\centering
\small
\setlength{\tabcolsep}{4pt}
\resizebox{\textwidth}{!}{%
\begin{tabular}{@{}llcccl@{}}
\toprule
Descriptor & Unit & Base & MMFF & Atom details & Lowest mean \\
\midrule
degrees & count & \nlvalpm{0.011615}{0.008813} {\scriptsize ($n=3$)} & \nlvalpm{0.020533}{0.023710} {\scriptsize ($n=3$)} & \nlvalpm{0.018701}{0.006194} {\scriptsize ($n=3$)} & Base \\
formal\_charges & e & \nlvalpm{0.003431}{0.000349} {\scriptsize ($n=3$)} & \nlvalpm{0.002960}{0.001328} {\scriptsize ($n=3$)} & \nlvalpm{0.004110}{0.000708} {\scriptsize ($n=3$)} & MMFF \\
graph\_connected\_components & count & \nlvalpm{0.000200}{0.000100} {\scriptsize ($n=3$)} & \nlvalpm{0.000167}{0.000058} {\scriptsize ($n=3$)} & \nlvalpm{0.000133}{0.000115} {\scriptsize ($n=3$)} & Atom details \\
graph\_num\_atoms & count & \nlvalpm{0.007147}{0.000686} {\scr